<a href="https://colab.research.google.com/github/Thashmikab/london-smart-meter-analysis/blob/main/notebooks/03_did_tou_change_behaviour.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 03 · Did time-of-use pricing change behaviour, and did it save money?

In 2013, 1,122 households in the Low Carbon London trial paid prices that changed through the day: a Normal rate of 11.76p/kWh most of the time, a High rate of 67.20p during announced price events and a Low rate of 3.99p at others. Everyone else stayed on a flat 14.228p. This notebook asks what that did.

**The answers, in plain terms**

- **Households did respond, a little.** Beyond a drop everyone showed in 2013 (largely a milder autumn), time-of-use households cut their electricity use by a further **1.8%**, about **65 kWh a year** each, and their evening-peak use (16:00–19:00) by **2.6%**.
- **Across many homes, that adds up.** For every 1,000 homes on the tariff, the evening peak was about **16 kW lower**, roughly five kettles switched off for the whole evening.
- **They didn't move a larger share of their use out of the evening.** The peak fell in step with everything else. The price events changed from day to day, so households responded to specific events rather than to a fixed evening window.
- **The tariff saved money, but not much:** about **£14 a year** for the average time-of-use household (median £10), roughly **3% of a £470 annual bill**. **87%** came out ahead; about 1 in 7 paid more than they would have on the flat rate.
- **Half the saving came from responding.** A household that ignored the prices would have saved only about 1.5% per kWh: the cheap Normal rate is mostly cancelled out by the expensive events. Responding cut a further **0.22p per kWh**, doubling the saving.

The bottom line: price signals did change behaviour, measurably, but the reward for responding was small. Larger price differences, or automation that responds on the household's behalf, would likely be needed for bigger shifts.

**Inputs:** `household_day` and `households` from notebook 02, read from Google Drive.

## Setup

In [ ]:
!pip install pyspark

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats
from google.colab import drive
from pyspark.sql import SparkSession
from pyspark.sql import functions as F, Window
from pyspark.sql.functions import col, count, when

## 1. Load the analysis tables and check they arrived as saved

A check, not more cleaning: the row counts, columns, nulls and flag counts should match what notebook 02 saved. The only nulls expected are `peak_share` on zero days, `holiday` on non-holidays and `cost_tou_gbp` outside 2013.

In [ ]:
#@title Start Spark and load the analysis tables from Drive

spark = SparkSession.builder.appName("london-smart-meters-analysis").getOrCreate()
spark.conf.set("spark.sql.session.timeZone", "UTC")

drive.mount("/content/drive")
DRIVE = "/content/drive/MyDrive/london-smart-meters"

household_day = spark.read.parquet(f"{DRIVE}/household_day")
households = spark.read.parquet(f"{DRIVE}/households")

In [ ]:
#@title Check the tables arrived as saved

print(f"household_day: {household_day.count():,} rows")     # 3,490,380
household_day.printSchema()

print(f"households: {households.count():,} rows")           # 5,560
households.printSchema()

# Nulls: only the expected ones should appear
household_day.select([count(when(col(c).isNull(), c)).alias(c) for c in household_day.columns]).show(vertical=True)

# The analysis flags, by group
households.groupBy("tariff").agg(
    F.count("*").alias("households"),
    F.sum(col("in_tariff_comparison").cast("int")).alias("in_tariff_comparison"),
    F.sum(col("in_2013_analysis").cast("int")).alias("in_2013_analysis")).show()

## 2. The comparison sample

Households flagged for the comparison (at least 60 usable days in each period), July–December of 2012 (before) and 2013 (prices in effect), zero days left out. Both periods cover the same months, so the season is the same.

In [ ]:
#@title Step 1: the comparison sample

sample = (household_day
    .filter(col("in_tariff_comparison") & ~col("zero_day"))
    .filter(col("year").isin(2012, 2013) & (col("month") >= 7))
    .withColumn("period", F.when(col("year") == 2012, "baseline_2012").otherwise("tariff_2013")))

sample.groupBy("tariff", "period").agg(
    F.countDistinct("household_id").alias("households"),
    F.count("*").alias("household_days")).orderBy("tariff", "period").show()

## 3. One row per household per period

**How these numbers are made.** Each household's daily total (its 48 half-hours added up in notebook 02) is averaged over that household's usable days in July–December. So `kwh` is the household's average electricity use **per day** in that period, and `peak_kwh` the average per day within 16:00–19:00. `peak_share` is the period's total peak kWh divided by its total kWh. Averaging per household first means every household counts once, whether it has 60 days or 184.

In [ ]:
#@title Step 2: average per household per period

hh_period = (sample.groupBy("household_id", "tariff", "period")
    .agg(F.avg("kwh").alias("kwh"),
         F.avg("kwh_peak").alias("peak_kwh"),
         (F.sum("kwh_peak") / F.sum("kwh")).alias("peak_share"),
         F.count("*").alias("days")))

# Put each household's two periods side by side
wide = (hh_period.groupBy("household_id", "tariff")
        .pivot("period", ["baseline_2012", "tariff_2013"])
        .agg(F.first("kwh").alias("kwh"),
             F.first("peak_kwh").alias("peak_kwh"),
             F.first("peak_share").alias("peak_share")))

pdf = wide.toPandas()
print(pdf.shape)          # 5,208 households
pdf.head()

## 4. Did time-of-use households change more than standard households?

**How the effect is measured.** For each household: its 2013 average minus its 2012 average. Then the time-of-use group's average change minus the standard group's average change. Anything that affected everyone between the two years, such as weather or energy prices in the news, appears in both changes and cancels out. What remains is the tariff's effect. The 95% confidence interval and p-value come from a two-sample t-test on the households' changes.

| Measure | ToU change | Std change | Effect | 95% CI | % of ToU baseline | p |
|---|---:|---:|---:|---|---:|---:|
| Daily kWh | −0.843 | −0.666 | **−0.177** | −0.343 to −0.010 | −1.8% | 0.038 |
| Peak kWh (16:00–19:00) | −0.197 | −0.150 | **−0.047** | −0.080 to −0.014 | −2.6% | 0.006 |
| Peak share | −0.0042 | −0.0031 | −0.0010 | −0.0025 to 0.0004 | −0.6% | 0.166 |

Time-of-use households cut total use and evening-peak use by more than standard households did. The share of use falling in the evening didn't change significantly: the evening fell in proportion with the rest of the day.

In [ ]:
#@title Step 3: did ToU households change more than standard households?

metrics = {"kwh": "Daily kWh", "peak_kwh": "Peak kWh (16:00-19:00)", "peak_share": "Peak share"}

# Each household's change from 2012 to 2013
for m in metrics:
    pdf[f"change_{m}"] = pdf[f"tariff_2013_{m}"] - pdf[f"baseline_2012_{m}"]

# The 2x2 table: group averages in each period
print("Group averages")
print(pdf.groupby("tariff")[[f"{p}_{m}" for m in metrics for p in ["baseline_2012", "tariff_2013"]]].mean().round(4).T)

# The effect: ToU's average change minus Std's average change, with a 95% confidence interval
rows = []
for m, label in metrics.items():
    tou = pdf.loc[pdf.tariff == "ToU", f"change_{m}"].dropna()
    std = pdf.loc[pdf.tariff == "Std", f"change_{m}"].dropna()
    effect = tou.mean() - std.mean()
    se = np.sqrt(tou.var() / len(tou) + std.var() / len(std))
    p = stats.ttest_ind(tou, std, equal_var=False).pvalue
    baseline = pdf.loc[pdf.tariff == "ToU", f"baseline_2012_{m}"].mean()
    rows.append({"measure": label,
                 "ToU change": round(tou.mean(), 4), "Std change": round(std.mean(), 4),
                 "effect": round(effect, 4),
                 "95% CI": f"{effect - 1.96*se:.4f} to {effect + 1.96*se:.4f}",
                 "effect as % of ToU baseline": round(100 * effect / baseline, 2),
                 "p-value": round(p, 4)})

pd.DataFrame(rows)

## 5. Why did everyone use less in 2013?

Standard households used about 6% less in July–December 2013 than a year earlier, without any price signals. Weather explains much of it: the second half of 2013 was **1.2 °C warmer** on average, with **16% fewer heating degree days** (609 against 727). Heating degree days add up how far each day's average temperature fell below 15.5 °C, the usual UK base for heating demand. Because both groups lived through the same weather, it doesn't affect the tariff comparison.

In [ ]:
#@title Was July-December 2013 milder than July-December 2012?

days = (household_day
    .select("day", "year", "month", "temp_mean").distinct()
    .filter(col("year").isin(2012, 2013) & (col("month") >= 7)))

days.groupBy("year").agg(
    F.round(F.avg("temp_mean"), 2).alias("avg_temp_C"),
    F.round(F.sum(F.greatest(F.lit(15.5) - col("temp_mean"), F.lit(0))), 0).alias("heating_degree_days")
).orderBy("year").show()

## 6. Did the tariff save money?

**How the saving is measured.** Every household's 2013 consumption was priced both ways in notebook 02: on the time-of-use schedule and at the flat rate. Comparing the **average price paid per kWh** removes the fact that time-of-use households use less overall. Standard households never saw the prices, so pricing their usage on the time-of-use schedule shows what the tariff does to a household that doesn't respond. The gap between the two groups is what responding was worth. Yearly savings are scaled to 365 days so households with part of a year are comparable.

| | Price per kWh | Saving per year (average / median) | Share better off |
|---|---:|---:|---:|
| Flat tariff | 14.228p | | |
| Standard households on the time-of-use schedule (no response) | 14.013p | £8.71 / £5.95 | 76% |
| Time-of-use households (responded) | 13.793p | £14.32 / £10.26 | 87% |

Time-of-use households paid **0.22p less per kWh** than non-responding households would have (95% CI 0.19 to 0.25p, p < 0.0001). Roughly half their saving came from the price structure and half from responding. In 2012 they had a slightly *higher* evening-peak share than standard households (0.183 against 0.179), so their underlying pattern was, if anything, more expensive on this schedule; the 0.22p gap is more likely understated than overstated.

In [ ]:
#@title Did the ToU tariff save money? (2013)

FLAT_P = 14.228   # pence per kWh, the standard tariff

# One row per household: total kWh and costs in 2013
bills = (household_day
    .filter(col("in_2013_analysis") & ~col("zero_day") & (col("year") == 2013))
    .groupBy("household_id", "tariff")
    .agg(F.sum("kwh").alias("kwh"),
         F.sum("cost_tou_gbp").alias("cost_tou"),
         F.sum("cost_flat_gbp").alias("cost_flat"),
         F.count("*").alias("days")))

b = bills.toPandas()

# Average price paid per kWh under the ToU schedule, in pence
b["price_tou_p"] = 100 * b["cost_tou"] / b["kwh"]

# Saving compared with the flat tariff, scaled to a full year
b["saving_per_year"] = (b["cost_flat"] - b["cost_tou"]) / b["days"] * 365

summary = b.groupby("tariff").agg(
    households=("household_id", "count"),
    avg_price_p=("price_tou_p", "mean"),
    saving_per_year_avg=("saving_per_year", "mean"),
    saving_per_year_median=("saving_per_year", "median"),
    share_who_saved=("saving_per_year", lambda s: (s > 0).mean()))
print(summary.round(3))

# Did responding lower the price paid? Compare ToU households with standard households
tou = b.loc[b.tariff == "ToU", "price_tou_p"]
std = b.loc[b.tariff == "Std", "price_tou_p"]
diff = tou.mean() - std.mean()
se = np.sqrt(tou.var() / len(tou) + std.var() / len(std))
p = stats.ttest_ind(tou, std, equal_var=False).pvalue
print(f"\nToU minus Std, average price: {diff:.3f} p/kWh "
      f"(95% CI {diff - 1.96*se:.3f} to {diff + 1.96*se:.3f}, p = {p:.4f})")

## 7. The results in one figure

Left: average daily use in each group, and where time-of-use households would have been had they followed the standard group's trend. Right: the average price per kWh under the flat tariff, under time-of-use prices with unchanged habits, and under time-of-use prices as households actually behaved. The split of the yearly saving uses each time-of-use household's own consumption: the price-structure part is the standard group's price gap applied to it, and the behaviour part is the rest.

In [ ]:
#@title Figure: the tariff's effect on use and on bills

import matplotlib.pyplot as plt

import matplotlib.pyplot as plt

def make_figure(std_12, std_13, tou_12, tou_13, flat_p, std_price, tou_price,
                saving_structure, saving_behaviour, path):
    expected = tou_12 + (std_13 - std_12)
    gap = tou_13 - expected
    blue, grey = "#2f6db3", "#8a8a8a"

    fig, (a, b) = plt.subplots(1, 2, figsize=(12.5, 5.2), gridspec_kw={"width_ratios": [1, 1.15]})

    # Panel A: change in daily use, July-December 2012 vs 2013
    x = [0, 1]
    a.plot(x, [std_12, std_13], "-o", color=grey, lw=2.5, ms=7, label="Standard tariff")
    a.plot(x, [tou_12, tou_13], "-o", color=blue, lw=2.5, ms=7, label="Time-of-use tariff")
    a.plot(x, [tou_12, expected], "--", color=blue, lw=1.5, alpha=0.6)
    a.plot([1], [expected], "o", mfc="white", mec=blue, ms=7)
    a.annotate("", xy=(1.05, tou_13), xytext=(1.05, expected),
               arrowprops=dict(arrowstyle="<->", color=blue, lw=1.2))
    a.text(1.11, (tou_13 + expected) / 2,
           f"{gap:+.2f} kWh/day ({100*gap/tou_12:+.1f}%):\nthe tariff's effect",
           va="center", fontsize=9.5, color=blue)
    a.text(1.06, expected + 0.02, f"{expected:.2f} if they had\nfollowed the standard group",
           color=blue, alpha=0.8, fontsize=8.5, ha="left", va="bottom")
    for v, xx, c in [(std_12, 0, grey), (std_13, 1, grey), (tou_12, 0, blue)]:
        a.text(xx - 0.06 if xx == 0 else xx + 0.06, v, f"{v:.2f}", color=c, fontsize=9,
               ha="right" if xx == 0 else "left", va="center")
    a.text(1, tou_13 - 0.05, f"{tou_13:.2f}", color=blue, fontsize=9, ha="center", va="top")
    a.set_xticks(x, ["Jul–Dec 2012\n(before)", "Jul–Dec 2013\n(prices in effect)"])
    a.set_xlim(-0.35, 1.85)
    a.set_ylim(min(tou_13, expected) - 0.2, max(std_12, std_13) + 0.15)
    a.set_ylabel("Average electricity use per household (kWh per day)")
    a.set_title("Everyone used less in 2013;\ntime-of-use households cut a little more", loc="left", fontsize=11.5)
    a.legend(frameon=False, loc="upper right", fontsize=9)

    # Panel B: price paid per kWh
    labels = ["Flat tariff", "Time-of-use prices,\nhabits unchanged", "Time-of-use prices,\nas households behaved"]
    vals = [flat_p, std_price, tou_price]
    ys = [2, 1, 0]
    b.plot(vals, ys, "o", color=blue, ms=10)
    for v, y in zip(vals, ys):
        b.text(v + 0.025, y + 0.18, f"{v:.2f}p", fontsize=10, ha="center", color="#222")
    b.set_yticks(ys, labels, fontsize=9.5)
    b.set_xlim(min(vals) - 0.15, flat_p + 0.12)
    b.set_ylim(-0.6, 2.7)
    b.set_xlabel("Average price paid per kWh (pence)")
    for (y1, y2, v1, v2, txt) in [(2, 1, flat_p, std_price, f"price structure:\n≈ £{saving_structure:.0f} a year"),
                                  (1, 0, std_price, tou_price, f"changed behaviour:\n≈ £{saving_behaviour:.0f} a year")]:
        b.annotate("", xy=(v2, y2 + 0.08), xytext=(v1, y1 - 0.08),
                   arrowprops=dict(arrowstyle="->", color="#c0392b", lw=1.3))
        b.text((v1 + v2) / 2 - 0.04, (y1 + y2) / 2, txt, fontsize=9, color="#c0392b", ha="right", va="center")
    b.set_title(f"Saving for a time-of-use household: about £{saving_structure + saving_behaviour:.0f} a year,\nhalf from the prices themselves, half from responding",
                loc="left", fontsize=11.5)
    for ax in (a, b):
        for s in ("top", "right"):
            ax.spines[s].set_visible(False)
    b.spines["left"].set_visible(False)
    b.tick_params(axis="y", length=0)

    fig.text(0.01, 0.01, "Low Carbon London trial: 1,062 time-of-use and 4,146 standard households (daily use); "
             "1,088 and 4,230 households in 2013 (prices).", fontsize=8, color="#666")
    fig.tight_layout(rect=(0, 0.03, 1, 1))
    fig.savefig(path, dpi=160)
    return fig


means = pdf.groupby("tariff")[["baseline_2012_kwh", "tariff_2013_kwh"]].mean()
std_price = b.loc[b.tariff == "Std", "price_tou_p"].mean()
tou_price = b.loc[b.tariff == "ToU", "price_tou_p"].mean()

tou_b = b[b.tariff == "ToU"]
annual_kwh = tou_b["kwh"] / tou_b["days"] * 365
saving_structure = ((FLAT_P - std_price) / 100 * annual_kwh).mean()
saving_behaviour = tou_b["saving_per_year"].mean() - saving_structure
print(f"Saving per year: £{saving_structure:.2f} price structure + £{saving_behaviour:.2f} behaviour")

!mkdir -p figures
make_figure(means.loc["Std", "baseline_2012_kwh"], means.loc["Std", "tariff_2013_kwh"],
            means.loc["ToU", "baseline_2012_kwh"], means.loc["ToU", "tariff_2013_kwh"],
            FLAT_P, std_price, tou_price, saving_structure, saving_behaviour,
            "figures/tou_effect_and_savings.png")
plt.show()

## What this means

- **Per household:** about 65 kWh less electricity a year (1.8%), a 2.6% smaller evening peak, and about £14 off the annual bill.
- **Per 1,000 homes:** the evening peak about 16 kW lower, roughly five kettles switched off for the whole evening, every evening.
- **Who gains:** most households (87%) came out ahead, but about 1 in 7 paid more than on the flat rate. A household that ignores the prices gains very little, because the cheap Normal rate is largely cancelled out by the expensive events.
- **The catch:** a 0.22p/kWh reward for responding is small. The response was real and measurable, but modest, which suggests price signals alone are a gentle lever; bigger price differences or automated response would be needed for larger shifts.

**Limits.** Time-of-use households volunteered, so they may differ from other households (in July–December 2012, before the tariff, they already used about 6% less); comparing each group's *change* from 2012 handles stable differences like that, but not differences in how the groups would have changed anyway. The 16:00–19:00 window is fixed, while price events moved from day to day, so the peak measures understate the direct response to individual events.

**Next:** notebook 04 asks whether the response cut carbon emissions, and what a larger shift of evening demand would have saved.